In [3]:
import os

BASE_URL = f"http://localhost:8000/v1"

os.environ["BASE_URL"]    = BASE_URL
os.environ["OPENAI_API_KEY"] = "abc-123"   

print("Config set:", BASE_URL)

Config set: http://localhost:8000/v1


In [8]:
pip install "pydantic-ai-slim[openai]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.1 MB/s  0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.3.0
    Uninstalling openai-2.3.0:
      Successfully uninstalled openai-2.3.0

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
!pip install -q pydantic-ai-slim openai


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [5]:
!curl http://localhost:8000/v1/models -H "Authorization: Bearer $OPENAI_API_KEY"

{"object":"list","data":[{"id":"Qwen2.5-7B-Instruct","object":"model","created":1781010253,"owned_by":"vllm","root":"Qwen/Qwen2.5-7B-Instruct","parent":null,"max_model_len":32768,"permission":[{"id":"modelperm-b80697d3d9a1427399d94ba91e9cbc92","object":"model_permission","created":1781010253,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [6]:
import os
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

provider = OpenAIProvider(
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

agent_model = OpenAIChatModel("Qwen2.5-7B-Instruct", provider=provider)

In [9]:
import os
import json
import uuid
import time
import random
from datetime import datetime, timedelta
from openai import OpenAI

# ── Config ────────────────────────────────────────────────────────────────────
BASE_URL   = "http://localhost:8000/v1"
API_KEY    = "abc-123"
MODEL      = "Qwen2.5-7B-Instruct"
OUTPUT     = "complaints.json"
TOTAL      = 80

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

# ── Category distribution ─────────────────────────────────────────────────────
CATEGORIES = {
    "fraud":           0.10,
    "card_dispute":    0.20,
    "net_banking_upi": 0.25,
    "kyc_onboarding":  0.10,
    "loan_emi":        0.15,
    "branch_atm":      0.08,
    "insurance":       0.07,
    "general":         0.05,
}

CHANNELS      = ["web", "email", "ivr", "branch"]
ACCOUNT_TYPES = ["savings", "current", "credit_card", "loan", "demat"]

# ── Per-category prompts ──────────────────────────────────────────────────────
PROMPTS = {
    "fraud": """Generate 1 realistic Indian bank customer complaint about fraud or unauthorized transaction.
Examples:
- "My HDFC credit card was used for Rs 8500 at a Delhi merchant I never visited. Block my card immediately."
- "Someone made 3 UPI transactions from my account totaling Rs 15000 last night while I was asleep."
Return ONLY the complaint text, nothing else.""",

    "card_dispute": """Generate 1 realistic Indian bank customer complaint about a credit or debit card issue.
Examples:
- "My SBI debit card got blocked after I tried paying at a petrol pump. Need it unblocked urgently."
- "Card transaction of Rs 2300 at Zomato is showing twice in my statement but I only ordered once."
Return ONLY the complaint text, nothing else.""",

    "net_banking_upi": """Generate 1 realistic Indian bank customer complaint about net banking or UPI failure.
Examples:
- "IMPS transfer of Rs 50000 debited from my account but not credited to beneficiary since 2 hours."
- "PhonePe UPI transaction failed but money got deducted. Transaction ref UPITXN99234."
Return ONLY the complaint text, nothing else.""",

    "kyc_onboarding": """Generate 1 realistic Indian bank customer complaint about KYC or account opening.
Examples:
- "My video KYC has failed 4 times even though my Aadhaar and PAN are clearly visible. Account still frozen."
- "Applied for savings account 3 weeks ago. Documents submitted but no update. Application ID APP2024567."
Return ONLY the complaint text, nothing else.""",

    "loan_emi": """Generate 1 realistic Indian bank customer complaint about loan EMI.
Examples:
- "EMI of Rs 12000 for personal loan bounced this month even though I had sufficient balance. Penalty charged."
- "Foreclosure request for home loan submitted 2 months ago. No NOC received yet. Loan account HL890234."
Return ONLY the complaint text, nothing else.""",

    "branch_atm": """Generate 1 realistic Indian bank customer complaint about branch or ATM.
Examples:
- "ATM at Bandra West dispensed Rs 1000 less but full Rs 5000 was debited. Ref no ATM20240312."
- "Branch manager at Dadar branch is refusing to update my nominee without valid reason."
Return ONLY the complaint text, nothing else.""",

    "insurance": """Generate 1 realistic Indian bank complaint about insurance claim or policy.
Examples:
- "Health insurance claim for hospitalization in January still pending after 3 months. Claim ID CLM78923."
- "Auto debit for LIC policy failed but policy shows lapsed. Amount was available in account."
Return ONLY the complaint text, nothing else.""",

    "general": """Generate 1 realistic Indian bank customer complaint about general banking service.
Examples:
- "Cheque book requested 3 weeks ago still not received at registered address."
- "Interest certificate for FD not generated on netbanking portal. Needed for ITR filing."
Return ONLY the complaint text, nothing else.""",
}

# ── Fallbacks if model unavailable ───────────────────────────────────────────
FALLBACKS = {
    "fraud":           "Unauthorized transaction of Rs 9500 on my credit card. Never made this purchase. Block immediately.",
    "card_dispute":    "Debit card declined at supermarket even though balance is sufficient.",
    "net_banking_upi": "UPI payment of Rs 25000 debited but receiver says not credited. Please refund urgently.",
    "kyc_onboarding":  "KYC documents uploaded 5 times but account still shows unverified.",
    "loan_emi":        "Extra EMI deducted this month. Account shows two debits of Rs 8500 each.",
    "branch_atm":      "ATM cash withdrawal short by Rs 500 but full amount debited. Malad West branch.",
    "insurance":       "Cashless claim rejected at network hospital without any reason. Policy is active.",
    "general":         "Unable to download account statement from net banking for last 3 months.",
}

# ── Priority + SLA + team maps ────────────────────────────────────────────────


# ── Generate one complaint text ───────────────────────────────────────────────
def generate_text(category, retries=3):
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": PROMPTS[category]}],
                temperature=0.85,
                max_tokens=120,
                timeout=30,
            )
            text = resp.choices[0].message.content.strip().strip('"').strip("'")
            if text.lower().startswith("complaint:"):
                text = text[10:].strip()
            return text
        except Exception as e:
            print(f"    attempt {attempt+1} failed: {e}")
            time.sleep(2)
    print(f"    using fallback for {category}")
    return FALLBACKS[category]

# ── Build one full record ─────────────────────────────────────────────────────
def build_record(category):
    created_at = datetime.now() - timedelta(
        hours=random.randint(0, 47),
        minutes=random.randint(0, 59)
    )

    return {
        "id":            str(uuid.uuid4()),
        "text":          generate_text(category),
        "category":      category,           # seed label for audit only
        "priority":      None,               # agent detects this
        "channel":       random.choice(CHANNELS),
        "account_type":  random.choice(ACCOUNT_TYPES),
        "sentiment":     None,               # agent detects this
        "status":        "pending",
        "assigned_team": None,               # router assigns this
        "created_at":    created_at.isoformat(),
        "sla_deadline":  None,               # router calculates this
    }

# ── Main ──────────────────────────────────────────────────────────────────────
# weighted category list
categories = random.choices(
    list(CATEGORIES.keys()),
    weights=list(CATEGORIES.values()),
    k=TOTAL
)

print(f"Generating {TOTAL} complaints using {MODEL} at {BASE_URL}\n")

complaints = []
for i, cat in enumerate(categories):
    print(f"[{i+1:>3}/{TOTAL}] {cat:<20}", end=" ", flush=True)
    record = build_record(cat)
    complaints.append(record)
    print(f"✓  {record['text'][:60]}...")

# save
with open(OUTPUT, "w") as f:
    json.dump(complaints, f, indent=2, default=str)

print(f"\nSaved {len(complaints)} complaints → {OUTPUT}")

# distribution summary
from collections import Counter
dist = Counter(c["category"] for c in complaints)
print("\nDistribution:")
for cat, cnt in dist.most_common():
    print(f"  {cat:<20} {cnt:>3}  {'█' * cnt}")

Generating 80 complaints using Qwen2.5-7B-Instruct at http://localhost:8000/v1

[  1/80] card_dispute         ✓  My HDFC credit card was declined at an ATM for an unauthoriz...
[  2/80] fraud                ✓  My SBI debit card was debited Rs 25,000 from an unknown loca...
[  3/80] general              ✓  The ATM card I applied for two months ago has still not been...
[  4/80] kyc_onboarding       ✓  My KYC process has been stuck for over a month now. I submit...
[  5/80] card_dispute         ✓  My HDFC debit card was declined at an ATM yesterday when I w...
[  6/80] card_dispute         ✓  My HDFC credit card was declined for an online purchase even...
[  7/80] insurance            ✓  I made a lump sum claim under my term insurance policy in Ma...
[  8/80] card_dispute         ✓  My HDFC credit card was charged twice for the same amount at...
[  9/80] loan_emi             ✓  EMI for my business loan of Rs 25000 bounced this month desp...
[ 10/80] insurance            ✓  Health insuran